# Desafio 2 · Estatística com Python · Serviço Meteorológico

Relatório estatístico sobre a **velocidade do vento** registrada pela rede internacional de estações meteorológicas. Base: `estacoes.csv` (tabela dimensão) e `leituras.csv` (tabela fato), disponíveis no repositório do desafio.

> Os dados são **fictícios**, gerados para treino. Os resultados mostram o método de análise e não descrevem o clima real.

## Pergunta de negócio

- **Pergunta:** a partir de qual velocidade o vento deve gerar um alerta à população, e em quais condições do tempo esse vento forte aparece?
- **Coluna de análise:** `velocidade_vento_kmh` (km/h), da tabela fato `leituras`.
- **Decisão que o resultado apoiaria:** definir o limite de velocidade para emitir alerta de vento forte e as situações em que o monitoramento das estações deve ser reforçado.

## Parte 1 · Carga e primeiras medidas

### Célula 01 · Importações

In [ ]:
# ---------- Bibliotecas: ferramentas prontas que o Python usa ----------
import warnings                    # esconde avisos técnicos para não distrair
from pathlib import Path           # manipula caminhos de arquivos e pastas
import shutil                      # manipula arquivos e pastas (ex.: mover, copiar, apagar)

# Estas linhas servem para trabalhar no Google Colab (conectar ao Drive e enviar arquivos)
from google.colab import drive     # conecta o Google Drive ao Colab (para ler e salvar arquivos)
from google.colab import files     # (files) envia arquivos do seu computador para o Colab

# Bibliotecas de análise de dados e de gráficos
import pandas as pd                # tabelas de dados (DataFrame)
import seaborn as sns              # gráficos bonitos e simples
import matplotlib.pyplot as plt    # ajustes nos gráficos (título, linhas...)

# ---------- Configurações gerais (valem para todos os gráficos abaixo) ----------
warnings.filterwarnings("ignore")           # esconde avisos técnicos para não distrair
sns.set_theme(style="whitegrid")            # deixa todos os gráficos com um visual limpo
plt.rcParams["figure.figsize"] = (8, 6)    # tamanho padrão dos gráficos (largura, altura)

### Célula 02 · Carregar os arquivos CSV

In [ ]:
# Nomes das tabelas do seu tema (veja a tabela acima), iguais aos nomes dos arquivos CSV.
NOME_TABELA_DIMENSAO = "estacoes"   # ex.: "livros"
NOME_TABELA_FATO = "leituras"           # ex.: "emprestimos"

# ---------- 1) Conectando ao Google Drive ----------
drive.mount("/content/drive")      # vai pedir autorização na primeira vez

# ---------- 2) Dizendo onde os arquivos estão guardados ----------
pasta_dados = Path("/content/drive/MyDrive/Colab Notebooks/desafios/dados")   # pasta onde a base fica guardada no Drive
pasta_dados.mkdir(parents=True, exist_ok=True)                                # cria a pasta "dados" se faltar

# O "/" junta a pasta com o nome do arquivo (o nome vem da variável de cada tabela).
caminho_dimensao = pasta_dados / f"{NOME_TABELA_DIMENSAO}.csv"
caminho_fato = pasta_dados / f"{NOME_TABELA_FATO}.csv"

# ---------- 3) Se faltar algum arquivo, envia os dois de uma vez e o código separa ----------
if not caminho_dimensao.exists() or not caminho_fato.exists():
    print(f"Selecione os arquivos CSV ({NOME_TABELA_DIMENSAO} e {NOME_TABELA_FATO}). Pode escolher os dois de uma vez:")
    enviados = files.upload()                                  # abre o seletor; aceita vários arquivos

    for nome_enviado in enviados:                              # percorre cada arquivo enviado
        if NOME_TABELA_DIMENSAO in nome_enviado.lower():       # o nome contém o da dimensão? salva como dimensão
            shutil.move(nome_enviado, caminho_dimensao)        # salva já com o nome da tabela
            print(f"{nome_enviado} -> {caminho_dimensao.name}")
        elif NOME_TABELA_FATO in nome_enviado.lower():         # o nome contém o da fato? salva como fato
            shutil.move(nome_enviado, caminho_fato)
            print(f"{nome_enviado} -> {caminho_fato.name}")
        else:                                                  # nome que não combina com nenhuma tabela
            print(f"Ignorado: {nome_enviado} (o nome não contém '{NOME_TABELA_DIMENSAO}' nem '{NOME_TABELA_FATO}')")

# Confere se os dois arquivos estão no Drive antes de ler.
for caminho in [caminho_dimensao, caminho_fato]:
    if not caminho.exists():
        raise FileNotFoundError(f"Falta o arquivo {caminho.name}. Execute a célula de novo e envie-o.")

# ---------- 4) Lendo os dois arquivos ----------
df_dimensao = pd.read_csv(caminho_dimensao)
df_fato = pd.read_csv(caminho_fato)

print(NOME_TABELA_DIMENSAO, "->", len(df_dimensao), "linhas")
print(NOME_TABELA_FATO, "->", len(df_fato), "linhas")

### Célula 03 · Conhecendo a tabela dimensão (`estacoes`)

In [ ]:
# Quantas linhas (registros do cadastro) e colunas (variáveis) tem a tabela dimensão?
# .shape devolve dois números: (quantidade de linhas, quantidade de colunas)
linhas, colunas = df_dimensao.shape
print(f"A tabela dimensão tem {linhas} linhas e {colunas} colunas (variáveis)")
display(df_dimensao.head())

### Célula 04 · Conhecendo a tabela fato (`leituras`)

In [ ]:
# Quantas linhas (eventos) e colunas (variáveis) tem a tabela fato?
# .shape devolve dois números: (quantidade de linhas, quantidade de colunas)
linhas, colunas = df_fato.shape
print(f"A tabela fato tem {linhas} linhas e {colunas} colunas (variáveis)")
display(df_fato.head())

### Célula 05 · Tipos das colunas

In [ ]:
# 1) Tipo de cada coluna e quantidade de valores preenchidos. Compare com o esquema do plano do tema.
df_fato.info()

### Célula 06 · Valores ausentes

In [ ]:
# 2) Valores ausentes por coluna (tabela fato).
print("\nValores ausentes por coluna:")
print(df_fato.isna().sum())

### Célula 07 · Linhas duplicadas

In [ ]:
# 3) Linhas duplicadas (tabela fato).
print("\nLinhas duplicadas:", df_fato.duplicated().sum())

### Célula 08 · Chave estrangeira

In [ ]:
# 4) Chave estrangeira: todos os registros da tabela fato devem apontar para um ID que existe na tabela dimensão.
COLUNA_FK = "estacao_id"        # ex.: "livro_id" (coluna da fato que referencia a dimensão)
COLUNA_ID = "estacao_id"   # ex.: "livro_id" (coluna de ID da dimensão)
print("\nChaves estrangeiras que batem:", df_fato[COLUNA_FK].isin(df_dimensao[COLUNA_ID]).all())

### Leitura da conferência

- A tabela fato `leituras` tem **62.000 linhas e 9 colunas**, e a tabela dimensão `estacoes` tem **10.050 linhas e 7 colunas**. Não há **nenhuma linha duplicada** na fato.
- A chave estrangeira bate com a dimensão (`True`): toda leitura aponta para uma estação que existe no cadastro.
- A coluna de análise `velocidade_vento_kmh` está como número decimal (`float64`), com os **62.000 valores preenchidos** e nenhum ausente.
- **Nenhuma coluna tem valor ausente.** As colunas `condicao_tempo` e `data_hora` chegam como texto (`object`), como previsto no esquema. A `data_hora` não é usada nos cálculos, por isso não precisa ser convertida para data.

### Célula 09 · Checar a coluna numérica

In [ ]:
# Coluna numérica escolhida no Passo 1 (nome EXATO, como está no esquema do tema).
COLUNA_NUMERICA = "velocidade_vento_kmh"

# A coluna escolhida, como uma série (uma coluna do DataFrame).
serie_numerica = df_fato[COLUNA_NUMERICA]
print("Valores preenchidos:", serie_numerica.count(), "| ausentes:", serie_numerica.isna().sum())

### Célula 10 · Medidas estatísticas

In [ ]:
# Medidas de Tendência central e dispersão.
medidas = pd.Series({
    "Média": serie_numerica.mean(),
    "Mediana": serie_numerica.median(),
    "Moda": serie_numerica.mode()[0],            # se houver empate, a primeira será selecionada
    "Desvio padrão": serie_numerica.std(),
    "Variância": serie_numerica.var(),
    "Mínimo": serie_numerica.min(),
    "Máximo": serie_numerica.max(),
    "Amplitude": serie_numerica.max() - serie_numerica.min(),
}).round(2)
display(medidas)

# Média muito diferente da mediana é sinal de valores extremos ou de dados assimétricos.
print("Média - mediana:", round(medidas["Média"] - medidas["Mediana"], 2))

### Frases de insight

- **A média de `velocidade_vento_kmh` é 11,94 km/h.** Numa leitura qualquer da rede, o vento é, em média, fraco a moderado. Esse número sozinho não indica perigo.

- **A mediana de `velocidade_vento_kmh` é 9,70 km/h.** Metade das leituras tem vento de até 9,7 km/h. A média fica **2,24 km/h acima** da mediana, sinal de que poucas leituras com vento muito forte puxam a média para cima. O vento típico da rede é o da mediana.

- **O desvio padrão de `velocidade_vento_kmh` é 9,25 km/h**, quase do tamanho da própria média. O vento varia muito de uma leitura para outra (de 0 a 157,4 km/h), então a média não serve para descrever as situações de risco.

- **Decisão final:** comunicar o vento típico pela **mediana** e definir o alerta de vento forte a partir dos **valores altos da distribuição**, e não a partir da média.

## Parte 2 · Análise completa

### Célula 11 · Juntar a fato com a dimensão

In [ ]:
# Junta a fato com a dimensão: cada registro da fato recebe as colunas do seu cadastro.
df_completo = df_fato.merge(df_dimensao, left_on=COLUNA_FK, right_on=COLUNA_ID, how="left", suffixes=("", "_dim"))

print("Linhas da fato:", len(df_fato), "| depois da junção:", len(df_completo))   # precisam ser iguais
df_completo.head()

### Célula 12 · Comparar grupos por uma coluna da fato (`condicao_tempo`)

In [ ]:
# Coluna categórica usada para comparar (da fato). Poucos valores diferentes funcionam melhor.
COLUNA_GRUPO = "condicao_tempo"

# Quantidade, média, mediana e desvio padrão de cada grupo, do maior para o menor valor médio.
por_grupo = df_completo.groupby(COLUNA_GRUPO)[COLUNA_NUMERICA].agg(["count", "mean", "median", "std"]).round(2)
por_grupo = por_grupo.sort_values("mean", ascending=False)
display(por_grupo)

### Leitura por `condicao_tempo`

- **`tempestade` tem a maior média: 27,88 km/h**, quase **3 vezes** a de `ensolarado` (10,02 km/h). A mediana confirma (23,4 km/h contra 8,5 km/h): não é um efeito de poucos valores extremos, o vento é mais forte na maior parte das leituras de tempestade.
- **`tempestade` também tem o maior desvio padrão: 20,17 km/h**, o dobro do de `chuvoso` (10,20 km/h). Na tempestade o vento é forte **e** imprevisível. `ensolarado` tem o menor desvio (7,01 km/h), o tempo mais estável.
- A ordem das médias acompanha a gravidade da condição: ensolarado (10,02) < nublado (12,02) < chuvoso (14,29) < tempestade (27,88). `tempestade` tem o menor grupo (1.848 leituras), mas ainda é uma quantidade suficiente para a comparação.

### Célula 13 · Comparar grupos por uma coluna da dimensão (`pais`)

In [ ]:
COLUNA_GRUPO = "pais"     # coluna da dimensão: só existe em df_completo por causa da junção

# Quantidade, média, mediana e desvio padrão de cada grupo, do maior para o menor valor médio.
por_grupo = df_completo.groupby(COLUNA_GRUPO)[COLUNA_NUMERICA].agg(["count", "mean", "median", "std"]).round(2)
por_grupo = por_grupo.sort_values("mean", ascending=False)
display(por_grupo)

### Leitura por `pais`

- A maior média é a da **França (12,15 km/h)** e a menor é a do **Chile (11,43 km/h)**: uma diferença de apenas **0,72 km/h** entre 15 países. As medianas também ficam muito próximas (de 9,3 a 9,9 km/h).
- Os desvios padrão vão de 8,69 km/h (Chile) a 9,62 km/h (Portugal): a variação do vento é parecida em todos os países.
- Como as médias são **quase iguais**, o país **não explica** a diferença de vento. O Chile tem o menor grupo (1.253 leituras), por isso a média um pouco menor pede cautela. O que separa o vento fraco do vento forte é a **condição do tempo**, e não o país da estação.

### Célula 14 · Regra do IQR

In [ ]:
# Faixa central dos dados: do 1º ao 3º quartil (25% e 75%).
q1 = serie_numerica.quantile(0.25)
q3 = serie_numerica.quantile(0.75)
iqr = q3 - q1

# Regra usual: fora de [Q1 - 1,5 x IQR ; Q3 + 1,5 x IQR] é candidato a outlier.
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Cálculo do número de outliers
sem_outliers = serie_numerica[(serie_numerica >= limite_inferior) & (serie_numerica <= limite_superior)]
qtd_outliers = serie_numerica.count() - sem_outliers.count()

print(f"Limites: {limite_inferior:.2f} a {limite_superior:.2f}")
print(f"Outliers: {qtd_outliers} de {serie_numerica.count()} valores")
print("Maiores valores:", serie_numerica.nlargest(5).tolist())
print("Menores valores:", serie_numerica.nsmallest(5).tolist())

### Célula 15 · Medidas com e sem os candidatos a outlier

In [ ]:
# Compara as medidas com e sem os candidatos a outlier.
comparacao = pd.DataFrame({
    "Com outliers": [serie_numerica.mean(), serie_numerica.median(), serie_numerica.std()],
    "Sem outliers": [sem_outliers.mean(), sem_outliers.median(), sem_outliers.std()],
}, index=["Média", "Mediana", "Desvio padrão"]).round(2)
display(comparacao)

### Célula 15 (complemento) · Em quais condições do tempo estão os outliers

Para decidir se os valores acima do limite são erro ou caso real, a célula mostra em quais condições do tempo eles aparecem e qual parte de cada condição passa do limite.

In [ ]:
# Marca as leituras acima do limite superior da regra do IQR.
acima_do_limite = df_completo[COLUNA_NUMERICA] > limite_superior

# Quantas leituras acima do limite existem em cada condição e qual o peso de cada uma no total.
outliers_por_condicao = pd.DataFrame({
    "Leituras acima do limite": df_completo[acima_do_limite]["condicao_tempo"].value_counts(),
    "% dos outliers": (df_completo[acima_do_limite]["condicao_tempo"].value_counts(normalize=True) * 100).round(1),
    "% das leituras da condição": (acima_do_limite.groupby(df_completo["condicao_tempo"]).mean() * 100).round(1),
})
display(outliers_por_condicao.sort_values("% das leituras da condição", ascending=False))

### Leitura e decisão registrada

- A regra do IQR marcou **2.300 das 62.000 leituras (3,7%)**, todas **acima de 31,25 km/h**. O limite inferior é negativo (−9,95 km/h), então nenhum valor baixo foi marcado. O maior valor é 157,4 km/h.
- Sem esses valores, a **média cai de 11,94 para 10,79 km/h** e o **desvio padrão cai de 9,25 para 6,87 km/h**, enquanto a **mediana quase não muda** (de 9,70 para 9,40 km/h). É o sinal clássico de outlier: poucos valores altos puxam a média e a dispersão.
- Os outliers seguem a lógica do tempo: **33,2% das leituras de `tempestade`** passam do limite, contra **1,2% das leituras de `ensolarado`**. Em número absoluto, porém, a maior parte deles (**1.015, ou 44,1%**) aparece em leituras `nublado`, que são metade da base.
- **Decisão: manter.** Vento acima de 31 km/h é plausível para o tema, se concentra nas condições mais severas e é justamente o evento que um serviço meteorológico precisa enxergar. Excluir esses valores esconderia os casos de risco.
- **Valores ausentes:** não há nenhum (Célula 06). Nada foi ignorado, preenchido ou removido.

### Célula 16 · Assimetria (skew)

In [ ]:
# Assimetria: perto de 0 = formato equilibrado; positiva = cauda para valores altos; negativa = para valores baixos.
print("Assimetria:", round(serie_numerica.skew(), 2))

### Célula 17 · Percentis

In [ ]:
# Percentis: por exemplo, 50% dos valores ficam na mediana ou abaixo dela.
print(serie_numerica.quantile([0.10, 0.25, 0.50, 0.75, 0.90]))

### Célula 18 · Quantidade de valores por faixa

In [ ]:
# Quantidade de valores em 5 faixas de mesmo tamanho (a versão em texto de um histograma).
display(pd.cut(serie_numerica, bins=5).value_counts().sort_index())

### Leitura da distribuição

- A **assimetria é 2,25**, bem positiva: a distribuição tem uma **cauda longa para a direita**, formada por poucas leituras de vento muito forte. Isso confirma a média (11,94 km/h) acima da mediana (9,70 km/h).
- **75% das leituras** têm vento de até **15,8 km/h** e **90%** têm vento de até **23,3 km/h**.
- A primeira faixa (até 31,5 km/h) concentra **59.758 leituras (96,4%)**. As faixas seguintes somam só 2.242 leituras: 2.088 entre 31,5 e 63 km/h, e **apenas 154 acima de 63 km/h**.
- **Em palavras:** a maioria das leituras fica entre 3 e 23 km/h, e poucas passam de 31 km/h.

## Recomendação

**Pergunta de negócio:** a partir de qual velocidade o vento deve gerar um alerta à população, e em quais condições do tempo esse vento forte aparece?

**Evidência 1:** o limite superior da regra do IQR é de **31,25 km/h**. Só **3,7% das leituras** (2.300 de 62.000) passam dele, e 90% das leituras ficam em até 23,3 km/h. O limite separa com clareza o vento comum do vento excepcional, sem gerar alerta em excesso.

**Evidência 2:** em `tempestade`, a média do vento é de **27,88 km/h**, quase 3 vezes a de `ensolarado` (10,02 km/h), e **1 em cada 3 leituras** (33,2%) passa do limite. Mesmo assim, **44,1% dos ventos fortes** acontecem com o tempo `nublado`, então a condição do tempo sozinha não basta para alertar.

**Recomendação em uma frase:** emitir alerta de vento forte sempre que uma leitura passar de **31 km/h**, em qualquer condição do tempo, e manter em **vigilância reforçada** as estações que registrarem tempestade, onde esse limite é ultrapassado com muito mais frequência.

**Limites da análise:** os dados são fictícios; os 2.300 outliers foram **mantidos** por serem casos plausíveis; não havia valores ausentes; as leituras não são periódicas (cada estação tem de 1 a 16 leituras), então a análise vale para a rede como um todo e não para uma estação específica; o país da estação não mostrou diferença relevante.

## Gráficos

### Célula 19 · Histograma

In [ ]:
# Histograma: como os valores se distribuem, com a curva da distribuição e as linhas de média e mediana.
sns.histplot(serie_numerica.dropna(), bins=30, kde=True)
plt.axvline(serie_numerica.mean(), color="red", linestyle="-", linewidth=2, label=f"Média: {serie_numerica.mean():.2f}")
plt.axvline(serie_numerica.median(), color="green", linestyle="--", linewidth=2, label=f"Mediana: {serie_numerica.median():.2f}")
plt.title(f"Distribuição de {COLUNA_NUMERICA}")
plt.xlabel("Velocidade do vento (km/h)")
plt.ylabel("Quantidade de leituras")
plt.legend()
plt.tight_layout()
plt.show()

### Célula 20 · Boxplot

In [ ]:
# Boxplot: a caixa é a faixa central (25% a 75%); os pontos fora são candidatos a outlier.
sns.boxplot(x=serie_numerica.dropna())
plt.axvline(serie_numerica.mean(), color="red", linestyle="-", linewidth=2, label=f"Média: {serie_numerica.mean():.2f}")
plt.title(f"Faixa central e outliers de {COLUNA_NUMERICA}")
plt.xlabel("Velocidade do vento (km/h)")
plt.legend()
plt.tight_layout()
plt.show()

### Célula 20 (complemento) · Boxplot por condição do tempo

In [ ]:
# Boxplot por condição do tempo, da mais calma para a mais severa, com a linha do limite de alerta.
ordem_condicoes = ["ensolarado", "nublado", "chuvoso", "tempestade"]
sns.boxplot(data=df_completo, x="condicao_tempo", y=COLUNA_NUMERICA, order=ordem_condicoes)
plt.axhline(limite_superior, color="red", linestyle="--", linewidth=2, label=f"Limite do IQR: {limite_superior:.2f} km/h")
plt.title("Velocidade do vento por condição do tempo")
plt.xlabel("Condição do tempo")
plt.ylabel("Velocidade do vento (km/h)")
plt.legend()
plt.tight_layout()
plt.show()

### Leitura dos gráficos

O histograma mostra a maior parte das leituras concentrada abaixo de 25 km/h, com uma cauda longa de rajadas à direita que puxa a média (linha vermelha) para depois da mediana (linha verde). O boxplot por condição mostra a caixa da `tempestade` bem acima das demais, com a parte de cima da caixa (o 3º quartil, perto de 38 km/h) acima do limite de 31,25 km/h, o que ilustra a recomendação: alertar pelo limite de velocidade e reforçar a vigilância nas tempestades.